notebooks/04_validation.ipynb

Purpose: Reproduce the plots and metrics from the server notebook and extend them across subjects and cameras.
Move from sam3d_mhr.ipynb:
- Cell 32: SAM/Vicon comparison plotting.
- Cell 34: geometric versus Euler comparisons.
- Cell 35: multi-camera overlay plots.
- Cells 37–42: validation metrics and batch reporting.
Existing module to extend: src/comparison_engine.py.

## Plotting

In [ ]:
# Load the SAM data 

# ==============================================================================
# EULER ANGLES — TIME ALIGNED TO VICON
# ==============================================================================

sam_df = pd.DataFrame(data_log).sort_values("ts").reset_index(drop=True)
sam_t = sam_df["ts"].values


def smooth_parametric_rotations(matrices, sequence, window=15, poly=3):
    r_objects = R.from_matrix(matrices)
    quats = r_objects.as_quat()

    for i in range(1, len(quats)):
        if np.dot(quats[i], quats[i - 1]) < 0:
            quats[i] = -quats[i]

    smoothed_quats = np.zeros_like(quats)
    for i in range(4):
        smoothed_quats[:, i] = savgol_filter(quats[:, i], window, poly)

    smoothed_quats /= np.linalg.norm(smoothed_quats, axis=1, keepdims=True)
    raw_euler = r_objects.as_euler(sequence, degrees=True)
    smoothed_euler = R.from_quat(smoothed_quats).as_euler(sequence, degrees=True)
    raw_euler = np.rad2deg(np.unwrap(np.deg2rad(raw_euler), axis=0))
    smoothed_euler = np.rad2deg(np.unwrap(np.deg2rad(smoothed_euler), axis=0))
    return raw_euler, smoothed_euler

# Extract the Euler angles for each joint and smooth them
sam_ankle_euler_raw, sam_ankle_euler_smooth = smooth_parametric_rotations(
    np.stack(sam_df["R_rankle_local"].values),
    sequence="xyz"
)
sam_ankle_raw = sam_ankle_euler_raw[:, 2]
sam_ankle_smooth = sam_ankle_euler_smooth[:, 2]

sam_knee_euler_raw, sam_knee_euler_smooth = smooth_parametric_rotations(
    np.stack(sam_df["R_rknee_local"].values),
    sequence="yxz"
)
sam_knee_raw = sam_knee_euler_raw[:, 2]
sam_knee_smooth = sam_knee_euler_smooth[:, 2]

sam_hip_euler_raw, sam_hip_euler_smooth = smooth_parametric_rotations(
    np.stack(sam_df["R_rhip_local"].values),
    sequence="xzy"
)
sam_hip_raw = sam_hip_euler_raw[:, 1]
sam_hip_smooth = sam_hip_euler_smooth[:, 1]

# ==============================================================================
# GEOMETRIC KEYPOINT ANGLES — TIME ALIGNED TO VICON
# ==============================================================================

sam_hip_geo_raw = hip_angle_sag
sam_hip_geo_smooth = hip_sag_smooth

sam_knee_geo_raw = knee_angle_sag
sam_knee_geo_smooth = knee_sag_smooth

# Keep ankle separate until its anatomical convention has been verified
sam_ankle_geo_raw = 180.0 - ankle_angle_sag
sam_ankle_geo_smooth = 180.0 - ankle_sag_smooth


# -------------------------------------------------------------------------------

print("Parametric Smoothing and Extraction Complete!")


# Align using the shared arm-raise event.
print("Aligning Signals via Master Arm-Raise Event...")
l_arm_vec = l_hand - l_sh
r_arm_vec = r_hand - r_sh
l_elev_vicon = np.sum(l_arm_vec * vicon_up, axis=0)
r_elev_vicon = np.sum(r_arm_vec * vicon_up, axis=0)
vicon_arm_raise = np.maximum(l_elev_vicon, r_elev_vicon)

vicon_sync_mask = vicon_t <= 4000.0
vicon_t_window = vicon_t[vicon_sync_mask]
vicon_arm_window = vicon_arm_raise[vicon_sync_mask]

sam_sync_mask = time_s <= 4.0
sam_t_window = time_s[sam_sync_mask] * 1000.0
sam_arm_window = max_arm_raise[sam_sync_mask]
f_interp_arm = interp1d(
    sam_t_window,
    sam_arm_window,
    bounds_error=False,
    fill_value="extrapolate"
)
sam_arm_resampled = f_interp_arm(vicon_t_window)


def normalize(sig):
    sig_centered = np.nan_to_num(sig - np.nanmean(sig))
    std = np.std(sig_centered)
    return sig_centered / std if std > 0 else sig_centered


a = normalize(vicon_arm_window)
b = normalize(sam_arm_resampled)
best_lag_frames = np.arange(-len(b) + 1, len(a))[
    np.argmax(correlate(a, b, mode="full"))
]
time_shift_ms = best_lag_frames * (1000.0 / vicon_fps)
sam_t_aligned = sam_t + time_shift_ms
print(f"-> Applied Global Time Shift from Arm Raise: {time_shift_ms:.1f} ms")



# Calibrate SAM angles against the initial static Vicon pose.
print("Applying Static Pose Offset Correction...")
STATIC_WINDOW_MS = 1500.0
vicon_static_mask = vicon_t < (vicon_t.min() + STATIC_WINDOW_MS)
sam_static_mask = sam_t_aligned < (sam_t_aligned.min() + STATIC_WINDOW_MS)

if np.nanmedian(sam_knee_smooth) < 0:
    sam_knee_raw *= -1
    sam_knee_smooth *= -1

hip_offset = np.nanmedian(vicon_hip_flex[vicon_static_mask]) - np.nanmedian(sam_hip_smooth[sam_static_mask])
sam_hip_raw_corr = sam_hip_raw + hip_offset
sam_hip_smooth_corr = sam_hip_smooth + hip_offset

knee_offset = np.nanmedian(vicon_knee_flex[vicon_static_mask]) - np.nanmedian(sam_knee_smooth[sam_static_mask])
sam_knee_raw_corr = sam_knee_raw + knee_offset
sam_knee_smooth_corr = sam_knee_smooth + knee_offset

ankle_offset = np.nanmedian(vicon_ankle_flex[vicon_static_mask]) - np.nanmedian(sam_ankle_smooth[sam_static_mask])
sam_ankle_raw_corr = sam_ankle_raw + ankle_offset
sam_ankle_smooth_corr = sam_ankle_smooth + ankle_offset


# Compare on a common timebase 
# Euler rotations and Vicon -----------------------------
common_t = np.linspace(
    max(sam_t_aligned.min(), vicon_t.min()),
    min(sam_t_aligned.max(), vicon_t.max()),
    200
)
f_vicon_a = interp1d(vicon_t, vicon_ankle_flex, bounds_error=False)
f_sam_smooth_a = interp1d(sam_t_aligned, sam_ankle_smooth_corr, bounds_error=False)
f_vicon_k = interp1d(vicon_t, vicon_knee_flex, bounds_error=False)
f_sam_smooth_k = interp1d(sam_t_aligned, sam_knee_smooth_corr, bounds_error=False)
f_vicon_h = interp1d(vicon_t, vicon_hip_flex, bounds_error=False)
f_sam_smooth_h = interp1d(sam_t_aligned, sam_hip_smooth_corr, bounds_error=False)

# Geometric/keypoint SAM curves ---------------
f_sam_geo_h = interp1d(sam_t_aligned, sam_hip_geo_smooth, bounds_error=False, fill_value=np.nan)
f_sam_geo_k = interp1d(sam_t_aligned, sam_knee_geo_smooth, bounds_error=False, fill_value=np.nan)
f_sam_geo_a = interp1d(sam_t_aligned, sam_ankle_geo_smooth, bounds_error=False, fill_value=np.nan)
# ---------------------------------------------

# print(f"ANKLE RMSE -> Smoothed: {np.sqrt(np.nanmean((f_vicon_a(common_t) - f_sam_smooth_a(common_t))**2)):.2f}°")
print(f"KNEE Euler RMSE -> Smoothed: {np.sqrt(np.nanmean((f_vicon_k(common_t) - f_sam_smooth_k(common_t))**2)):.2f}°")
print(f"HIP Euler RMSE  -> Smoothed: {np.sqrt(np.nanmean((f_vicon_h(common_t) - f_sam_smooth_h(common_t))**2)):.2f}°")
print(f"KNEE Geometric RMSE -> Smoothed: {np.sqrt(np.nanmean((f_vicon_k(common_t) - f_sam_geo_k(common_t))**2)):.2f}°")
print(f"HIP Geometric RMSE  -> Smoothed: {np.sqrt(np.nanmean((f_vicon_h(common_t) - f_sam_geo_h(common_t))**2)):.2f}°")
print(f'KNEE Geometric RMSE -> Raw: {np.sqrt(np.nanmean((f_vicon_k(common_t) - f_sam_geo_k(common_t))**2)):.2f}°')
print(f'HIP Geometric RMSE  -> Raw: {np.sqrt(np.nanmean((f_vicon_h(common_t) - f_sam_geo_h(common_t))**2)):.2f}°')


print("Exporting biomechanical angles to CSV...")
export_df = pd.DataFrame({
    "time_ms_aligned": sam_t_aligned,
    "vicon_ankle_flex": f_vicon_a(sam_t_aligned),
    "vicon_knee_flex": f_vicon_k(sam_t_aligned),
    "vicon_hip_flex": f_vicon_h(sam_t_aligned),
    "r_ankle_flex_euler": sam_ankle_smooth_corr,
    "r_knee_flex_euler": sam_knee_smooth_corr,
    "r_hip_flex_euler": sam_hip_smooth_corr, 
    "r_ankle_flex_geo": sam_ankle_geo_smooth,
    "r_knee_flex_geo": sam_knee_geo_smooth,
    "r_hip_flex_geo": sam_hip_geo_smooth

    
})
export_df.to_csv(EXPORT_CSV_PATH, index=False)
print(f"Data saved to: {EXPORT_CSV_PATH}")


plt.figure(figsize=(14, 6))

plt.plot(
    vicon_t / 1000,
    vicon_knee_flex,
    label="Vicon",
    linewidth=3
)

plt.plot(
    sam_t_aligned / 1000,
    sam_knee_smooth_corr,
    label="MHR Euler",
    linewidth=2
)

plt.plot(
    sam_t_aligned / 1000,
    sam_knee_geo_smooth,
    label="MHR Geometric",
    linewidth=2
)

plt.plot(
    sam_t_aligned / 1000,
    sam_knee_geo_raw,
    label="MHR Geometric",
    linewidth=2
)

plt.xlabel("Time (s)")
plt.ylabel("Knee Flexion (degrees)")
plt.title("Knee Flexion: Vicon vs MHR Euler vs MHR Geometric")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# PLOT PARAMETRIC MHR vs GEOMETRIC KEYPOINTS vs VICON
# ==============================================================================

for joint_name, vicon_func, sam_euler_raw, sam_euler_smooth, \
        sam_geo_raw, sam_geo_smooth, ylabel in [

    (
        "Knee",
        f_vicon_k,
        sam_knee_raw_corr,
        sam_knee_smooth_corr,
        sam_knee_geo_raw,
        sam_knee_geo_smooth,
        "Knee Flexion Angle (degrees)"
    ),

    (
        "Hip",
        f_vicon_h,
        sam_hip_raw_corr,
        sam_hip_smooth_corr,
        sam_hip_geo_raw,
        sam_hip_geo_smooth,
        "Hip Flexion Angle (degrees)"
    ),

    (
        "Ankle",
        f_vicon_a,
        sam_ankle_raw_corr,
        sam_ankle_smooth_corr,
        sam_ankle_geo_raw,
        sam_ankle_geo_smooth,
        "Ankle Angle (degrees)"
    )
]:

    plt.figure(figsize=(14, 6))

    # ------------------------------------------------------------------
    # VICON
    # ------------------------------------------------------------------
    plt.plot(
        common_t / 1000.0,
        vicon_func(common_t),
        label="Vicon Gold Standard",
        color="black",
        linewidth=2.5,
        zorder=4
    )

    # ------------------------------------------------------------------
    # PARAMETRIC MHR / EULER
    # ------------------------------------------------------------------
    plt.plot(
        sam_t_aligned / 1000.0,
        sam_euler_raw,
        label="SAM 3D — Parametric MHR Raw",
        color="tab:orange",
        linewidth=1.2,
        alpha=0.35,
        zorder=1
    )

    plt.plot(
        sam_t_aligned / 1000.0,
        sam_euler_smooth,
        label="SAM 3D — Parametric MHR Smoothed",
        color="tab:orange",
        linewidth=2.0,
        alpha=0.9,
        zorder=2
    )

    # ------------------------------------------------------------------
    # GEOMETRIC / KEYPOINT
    # ------------------------------------------------------------------
    plt.plot(
        sam_t_aligned / 1000.0,
        sam_geo_raw,
        label="SAM 3D — Geometric Raw",
        color="tab:green",
        linewidth=1.2,
        alpha=0.35,
        zorder=1
    )

    plt.plot(
        sam_t_aligned / 1000.0,
        sam_geo_smooth,
        label="SAM 3D — Geometric Smoothed",
        color="tab:green",
        linewidth=2.5,
        alpha=0.9,
        zorder=3
    )

    plt.xlabel("Time (seconds)")
    plt.ylabel(ylabel)
    plt.title(
        f"Validation: {joint_name} — "
        "Vicon vs Parametric MHR vs Geometric Keypoints"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
 
# --- 1. CONFIGURATION ---
CSV_CAM0 = "/app/data/s9_lunge_cam0_joint_angles.csv"
CSV_CAM1 = "/app/data/s9_lunge_cam1_joint_angles.csv"
CSV_CAM2 = "/app/data/s9_lunge_cam2_joint_angles.csv"

# --- 2. LOAD DATA ---
df_c0 = pd.read_csv(CSV_CAM0) if os.path.exists(CSV_CAM0) else None
df_c1 = pd.read_csv(CSV_CAM1) if os.path.exists(CSV_CAM1) else None
df_c2 = pd.read_csv(CSV_CAM2) if os.path.exists(CSV_CAM2) else None

# --- 3. MULTI-CAMERA PLOTTING FUNCTION ---
def plot_multi_camera_comparison(joint_name, vicon_col, sam_smooth_col):
    plt.figure(figsize=(14, 6))
    
    # We only need the Vicon data from one dataframe, assuming they share the same timeline
    if df_c0 is not None and vicon_col in df_c0.columns:
        time_s = df_c0['time_ms_aligned'] / 1000.0
        plt.plot(time_s, df_c0[vicon_col], label='Vicon Gold Standard', color='black', linewidth=3, zorder=5)

    # Plot Camera 0 (e.g., Front/Side View)
    if df_c0 is not None and sam_smooth_col in df_c0.columns:
        time_c0 = df_c0['time_ms_aligned'] / 1000.0
        plt.plot(time_c0, df_c0[sam_smooth_col], label='SAM 3D (Cam 0)', color='tab:blue', linewidth=2, alpha=0.8)

    # Plot Camera 1 
    if df_c1 is not None and sam_smooth_col in df_c1.columns:
        time_c1 = df_c1['time_ms_aligned'] / 1000.0
        plt.plot(time_c1, df_c1[sam_smooth_col], label='SAM 3D (Cam 1)', color='tab:orange', linewidth=2, alpha=0.8)

    # Plot Camera 2 (e.g., Rear View)
    if df_c2 is not None and sam_smooth_col in df_c2.columns:
        time_c2 = df_c2['time_ms_aligned'] / 1000.0
        plt.plot(time_c2, df_c2[sam_smooth_col], label='SAM 3D (Cam 2)', color='tab:green', linewidth=2, alpha=0.8)

    # Formatting
    plt.xlabel('Time (seconds)')
    plt.ylabel(f'{joint_name} Flexion Angle (degrees)')
    plt.title(f'Multi-Camera Angle Comparison: {joint_name} Flexion')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# --- 4. GENERATE COMPARISON PLOTS ---
plot_multi_camera_comparison('Knee', 'vicon_knee_flex', 'r_knee_flex_smooth')
plot_multi_camera_comparison('Hip', 'vicon_hip_flex', 'r_hip_flex_smooth')

## Validation metrics

In [ ]:
from scipy.stats import pearsonr


def calculate_validation_metrics(vicon, sam, time_ms):
    """
    Compare SAM/MHR angle with Vicon on the same timestamps.
    """

    vicon = np.asarray(vicon, dtype=float)
    sam = np.asarray(sam, dtype=float)
    time_ms = np.asarray(time_ms, dtype=float)

    valid = (
        np.isfinite(vicon) &
        np.isfinite(sam) &
        np.isfinite(time_ms)
    )

    v = vicon[valid]
    s = sam[valid]
    t = time_ms[valid]

    if len(v) < 3:
        return None

    # ------------------------------
    # Raw comparison
    # ------------------------------

    diff = s - v

    rmse_raw = np.sqrt(np.mean(diff**2))
    mae_raw = np.mean(np.abs(diff))
    bias = np.mean(diff)

    r = pearsonr(v, s)[0]

    # ------------------------------
    # ROM
    # ------------------------------

    vicon_rom = np.max(v) - np.min(v)
    sam_rom = np.max(s) - np.min(s)
    rom_error = sam_rom - vicon_rom

    # ------------------------------
    # Peak angle
    # ------------------------------

    vicon_peak = np.max(v)
    sam_peak = np.max(s)

    peak_error = sam_peak - vicon_peak

    # ------------------------------
    # Peak timing
    # ------------------------------

    vicon_peak_time = t[np.argmax(v)]
    sam_peak_time = t[np.argmax(s)]

    peak_time_error_ms = sam_peak_time - vicon_peak_time

    # ------------------------------
    # Constant-offset correction
    # ------------------------------

    offset = np.median(v - s)
    s_corrected = s + offset

    rmse_offset = np.sqrt(
        np.mean((s_corrected - v)**2)
    )

    mae_offset = np.mean(
        np.abs(s_corrected - v)
    )

    return {
        "rmse_raw_deg": rmse_raw,
        "mae_raw_deg": mae_raw,
        "bias_deg": bias,

        "rmse_offset_deg": rmse_offset,
        "mae_offset_deg": mae_offset,
        "offset_deg": offset,

        "pearson_r": r,

        "vicon_rom_deg": vicon_rom,
        "sam_rom_deg": sam_rom,
        "rom_error_deg": rom_error,

        "vicon_peak_deg": vicon_peak,
        "sam_peak_deg": sam_peak,
        "peak_error_deg": peak_error,

        "peak_time_error_ms": peak_time_error_ms,

        "n_samples": len(v)
    }

In [ ]:
from scipy.interpolate import interp1d

# Interpolate Vicon angles to SAM timestamps for comparison
f_vicon_hip = interp1d(
    vicon_t,
    vicon_hip_flex,
    bounds_error=False,
    fill_value=np.nan
)

f_vicon_knee = interp1d(
    vicon_t,
    vicon_knee_flex,
    bounds_error=False,
    fill_value=np.nan
)

f_vicon_ankle = interp1d(
    vicon_t,
    vicon_ankle_flex,
    bounds_error=False,
    fill_value=np.nan
)


vicon_hip_on_sam = f_vicon_hip(sam_t_aligned)
vicon_knee_on_sam = f_vicon_knee(sam_t_aligned)
vicon_ankle_on_sam = f_vicon_ankle(sam_t_aligned)

In [ ]:
validation_df = pd.DataFrame()

In [ ]:
subject = "S9"
trial = "lunge_01"

results = []

for camera in ["cam0", "cam1", "cam2"]:
    csv_path = f"/app/data/{subject.lower()}_lunge_{camera}_joint_angles.csv"

    result = validate_angle_csv(
        csv_path,
        subject=subject,
        trial=trial,
        camera=camera,
        method="geo"
    )
    results.append(result)

new_results = pd.concat(results, ignore_index=True)

validation_df = pd.concat(
    [validation_df, new_results],
    ignore_index=True
)

display(validation_df)
